In [6]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.stats import pointbiserialr
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Add src to path if needed
project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import load_raw_data, RAW_DATA_PATH

df_raw = load_raw_data(RAW_DATA_PATH)
print(f"Loaded {len(df_raw)} customers.")

Loaded 4372 customers.


In [7]:
# Define exploration themes
THEMES = {
    "Friction": ["ReturnRatio", "CancelledTransactions", "NegativeQuantityCount", "SupportTicketsCount"],
    "Explorer": ["UniqueProducts", "UniqueDescriptions", "AvgProductsPerTransaction", "UniqueCountries"],
    "Timing": ["PreferredDayOfWeek", "PreferredHour", "PreferredMonth", "WeekendPurchaseRatio", "AvgDaysBetweenPurchases"],
    "Basket": ["AvgQuantityPerTransaction", "AvgLinesPerInvoice", "AvgProductsPerTransaction", "MonetaryAvg", "MonetaryStd"],
    "SpendVolatility": ["MonetaryStd", "MonetaryAvg", "MonetaryMax"],
    "Lifecycle": ["CustomerTenureDays", "FirstPurchaseDaysAgo", "TotalTransactions", "SatisfactionScore"]
}

# Pre-clean sentinel values for Support Tickets before exploration
if "SupportTicketsCount" in df_raw.columns:
    df_raw["SupportTicketsCount"] = df_raw["SupportTicketsCount"].replace([-1, 999], np.nan)

In [8]:
def select_and_scale(df: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    available = [f for f in features if f in df.columns]
    sub = df[available].copy()
    
    # Median imputation
    for col in sub.columns:
        if sub[col].isnull().any():
            sub[col] = sub[col].fillna(sub[col].median())
            
    # Remove pathological outliers before K-Means via 1st-99th percentile clipping
    for col in sub.columns:
        lo, hi = sub[col].quantile(0.01), sub[col].quantile(0.99)
        if pd.notna(lo) and pd.notna(hi):
            sub[col] = sub[col].clip(lo, hi)
        
    scaler = StandardScaler()
    scaled = pd.DataFrame(scaler.fit_transform(sub), columns=available, index=sub.index)
    return sub, scaled

In [ ]:
results = []
cluster_details = {}

for name, features in THEMES.items():
    try:
        sub, scaled = select_and_scale(df_raw, features)
        
        best_sil, best_k, best_labels = -1, 0, None
        for k in range(2, 6):
            km = KMeans(n_clusters=k, n_init=10, random_state=42)
            labels = km.fit_predict(scaled)
            sil = silhouette_score(scaled, labels)
            if sil > best_sil:
                best_sil, best_k, best_labels = sil, k, labels
                
        # Correlation with Churn
        cc = 0.0
        if "Churn" in df_raw.columns:
            churn_target = df_raw.loc[scaled.index, "Churn"]
            r, _ = pointbiserialr(best_labels, churn_target)
            cc = abs(r) if not np.isnan(r) else 0.0
            
        results.append({
            "Theme": name,
            "Optimal k": best_k,
            "Silhouette": best_sil,
            "Churn Corr (|r|)": cc,
            "Score (Sil - Corr)": best_sil - cc
        })
        
        # Save cluster profiles for the optimal K
        profile_df = sub.copy()
        profile_df["Cluster"] = best_labels
        if "Churn" in df_raw.columns:
            profile_df["Churn"] = churn_target
            
        details = profile_df.groupby("Cluster").mean()
        details.insert(0, "Size", profile_df.groupby("Cluster").size())
        cluster_details[name] = details
        
    except Exception as e:
        print(f"Skipping {name} due to error: {e}")

summary_df = pd.DataFrame(results).sort_values("Score (Sil - Corr)", ascending=False)
display(summary_df)

In [ ]:
# View details for the top-scoring combinations
for theme in summary_df.head(3)["Theme"]:
    print(f"\n{'='*60}\nTHEME: {theme}\n{'='*60}")
    display(cluster_details[theme])


THEME: SpendVolatility


,Size,MonetaryStd,MonetaryAvg,MonetaryMax,Churn
Cluster,,,,,
0,4194,16.187811,20.544927,80.205846,0.332141
1,178,166.332834,172.781600,771.267640,0.342697



THEME: Basket


,Size,AvgQuantityPerTransaction,AvgLinesPerInvoice,AvgProductsPerTransaction,MonetaryAvg,MonetaryStd,Churn
Cluster,,,,,,,
0,4182,11.064682,20.270750,16.978542,19.424506,17.080055,0.329986
1,190,106.609585,4.245305,2.912918,187.827712,137.211239,0.389474



THEME: Friction


,Size,ReturnRatio,CancelledTransactions,NegativeQuantityCount,SupportTicketsCount,Churn
Cluster,,,,,,
0,341,0.193279,13.263226,13.263226,2.032258,0.255132
1,4031,0.013952,0.800050,0.800050,1.955098,0.339122
